# Notebook 5: AI Agent, Recommendations & Ethical Analysis
## AI Agent for Disease Risk Awareness and Prevention

This notebook covers:
- **AI Agent Design:** Risk classification (Low/Medium/High)
- **Supervised Model Integration:** Uses probability outputs from Notebook 3
- **Recommendation System:** Rule-based + model-driven guidance
- **Ethical Analysis:** Bias detection, fairness metrics, demographic parity
- **Privacy Considerations:** Data handling, consent, transparency
- **Summary & Deployment Strategy**

**Evaluation Rubric:** AI Agent Implementation (10 marks), Ethical Analysis (5 marks)

## 1. Setup and Load Supervised Models

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_style('whitegrid')

# Load data and models
pima_X_train = pd.read_csv('../data/pima_X_train_engineered.csv')
pima_X_test = pd.read_csv('../data/pima_X_test_engineered.csv')
pima_y_test = pd.read_csv('../data/pima_y_test.csv').values.ravel()

heart_X_train = pd.read_csv('../data/heart_X_train_engineered.csv')
heart_X_test = pd.read_csv('../data/heart_X_test_engineered.csv')
heart_y_test = pd.read_csv('../data/heart_y_test.csv').values.ravel()

with open('../models/pima_best_model.pkl', 'rb') as f:
    pima_model = pickle.load(f)
with open('../models/heart_best_model.pkl', 'rb') as f:
    heart_model = pickle.load(f)

print("✓ Models and data loaded successfully")

✓ Models and data loaded successfully


## 2. AI Agent Design & Risk Classification

In [2]:
class DiseaseRiskAgent:
    """AI Agent for Disease Risk Assessment and Prevention Recommendations"""
    
    def __init__(self, model, dataset_type='pima'):
        self.model = model
        self.dataset_type = dataset_type
        self.risk_thresholds = {'low': 0.3, 'medium': 0.7}  # prob < 0.3: Low, 0.3-0.7: Medium, > 0.7: High
    
    def classify_risk(self, sample_proba):
        """Classify risk level based on model probability"""
        if sample_proba < self.risk_thresholds['low']:
            return 'Low Risk'
        elif sample_proba < self.risk_thresholds['medium']:
            return 'Medium Risk'
        else:
            return 'High Risk'
    
    def get_recommendations(self, sample, risk_level):
        """Generate personalized recommendations based on risk level"""
        recommendations = {
            'Low Risk': [
                '✓ Maintain current lifestyle with regular exercise (150 min/week)',
                '✓ Continue balanced diet monitoring',
                '✓ Annual health checkups recommended'
            ],
            'Medium Risk': [
                '⚠ Increase physical activity to 200 min/week',
                '⚠ Reduce sugar and processed food intake',
                '⚠ Schedule quarterly health checkups',
                '⚠ Monitor weight and blood pressure regularly'
            ],
            'High Risk': [
                '🔴 URGENT: Schedule appointment with healthcare provider',
                '🔴 Intensive lifestyle modification program required',
                '🔴 Monthly health monitoring recommended',
                '🔴 Consider medication consultation with physician',
                '🔴 Join support groups for disease prevention'
            ]
        }
        return recommendations.get(risk_level, [])
    
    def predict_and_recommend(self, sample):
        """Complete prediction and recommendation pipeline"""
        proba = self.model.predict_proba([sample])[0, 1]
        risk_level = self.classify_risk(proba)
        recommendations = self.get_recommendations(sample, risk_level)
        
        return {
            'probability': round(proba, 4),
            'risk_level': risk_level,
            'recommendations': recommendations
        }

# Initialize agents
pima_agent = DiseaseRiskAgent(pima_model, 'pima')
heart_agent = DiseaseRiskAgent(heart_model, 'heart')

print("✓ AI Agents initialized successfully")
print("\nAgent Configuration:")
print(f"  Low Risk threshold: < {pima_agent.risk_thresholds['low']}")
print(f"  Medium Risk threshold: {pima_agent.risk_thresholds['low']} - {pima_agent.risk_thresholds['medium']}")
print(f"  High Risk threshold: > {pima_agent.risk_thresholds['medium']}")

✓ AI Agents initialized successfully

Agent Configuration:
  Low Risk threshold: < 0.3
  Medium Risk threshold: 0.3 - 0.7
  High Risk threshold: > 0.7


## 3. Agent Demonstration - Sample Predictions & Recommendations

In [3]:
# Demonstrate agent on test samples
print("\n" + "="*70)
print("PIMA DATASET - AI AGENT RECOMMENDATIONS")
print("="*70)

# Use dynamic indices based on dataset size
pima_n_samples = len(pima_X_test)
pima_sample_indices = [0, pima_n_samples // 2, pima_n_samples - 1]

for idx in pima_sample_indices:
    sample = pima_X_test.iloc[idx].values
    result = pima_agent.predict_and_recommend(sample)
    actual = "Disease" if pima_y_test[idx] == 1 else "No Disease"
    
    print(f"\nSample {idx}:")
    print(f"  Risk Probability: {result['probability']}")
    print(f"  Classifications: {result['risk_level']} | Actual: {actual}")
    print(f"  Recommendations:")
    for rec in result['recommendations']:
        print(f"    {rec}")

print("\n" + "="*70)
print("HEART DISEASE DATASET - AI AGENT RECOMMENDATIONS")
print("="*70)

# Use dynamic indices based on dataset size
heart_n_samples = len(heart_X_test)
heart_sample_indices = [0, heart_n_samples // 2, heart_n_samples - 1]

for idx in heart_sample_indices:
    sample = heart_X_test.iloc[idx].values
    result = heart_agent.predict_and_recommend(sample)
    actual = "Disease" if heart_y_test[idx] == 1 else "No Disease"
    
    print(f"\nSample {idx}:")
    print(f"  Risk Probability: {result['probability']}")
    print(f"  Classification: {result['risk_level']} | Actual: {actual}")
    print(f"  Recommendations:")
    for rec in result['recommendations']:
        print(f"    {rec}")


PIMA DATASET - AI AGENT RECOMMENDATIONS

Sample 0:
  Risk Probability: 0.3578
  Classifications: Medium Risk | Actual: Disease
  Recommendations:
    ⚠ Increase physical activity to 200 min/week
    ⚠ Reduce sugar and processed food intake
    ⚠ Schedule quarterly health checkups
    ⚠ Monitor weight and blood pressure regularly

Sample 33:
  Risk Probability: 0.1204
  Classifications: Low Risk | Actual: No Disease
  Recommendations:
    ✓ Maintain current lifestyle with regular exercise (150 min/week)
    ✓ Continue balanced diet monitoring
    ✓ Annual health checkups recommended

Sample 66:
  Risk Probability: 0.5265
  Classifications: Medium Risk | Actual: Disease
  Recommendations:
    ⚠ Increase physical activity to 200 min/week
    ⚠ Reduce sugar and processed food intake
    ⚠ Schedule quarterly health checkups
    ⚠ Monitor weight and blood pressure regularly

HEART DISEASE DATASET - AI AGENT RECOMMENDATIONS

Sample 0:
  Risk Probability: 0.6567
  Classification: Medium Risk 

## 4. Ethical Analysis - Bias Detection & Fairness

In [4]:
print("\n" + "="*70)
print("ETHICAL ANALYSIS - FAIRNESS & BIAS DETECTION")
print("="*70)

def calculate_fairness_metrics(y_true, y_pred, sensitive_feature):
    """Calculate fairness metrics: demographic parity, equalized odds"""
    
    # Split by sensitive feature
    group_0 = sensitive_feature == 0
    group_1 = sensitive_feature == 1
    
    # Positive prediction rates (Demographic Parity)
    ppr_0 = np.mean(y_pred[group_0]) if np.sum(group_0) > 0 else 0
    ppr_1 = np.mean(y_pred[group_1]) if np.sum(group_1) > 0 else 0
    demographic_parity_diff = abs(ppr_0 - ppr_1)
    
    # True Positive Rates (Equalized Odds) - handle cases where no positive samples exist
    group_0_pos = group_0 & (y_true == 1)
    group_1_pos = group_1 & (y_true == 1)
    
    tpr_0 = np.mean(y_pred[group_0_pos]) if np.sum(group_0_pos) > 0 else 0
    tpr_1 = np.mean(y_pred[group_1_pos]) if np.sum(group_1_pos) > 0 else 0
    equalized_odds_diff = abs(tpr_0 - tpr_1)
    
    return {
        'demographic_parity_diff': round(demographic_parity_diff, 4),
        'equalized_odds_diff': round(equalized_odds_diff, 4),
        'ppr_group_0': round(ppr_0, 4),
        'ppr_group_1': round(ppr_1, 4),
        'tpr_group_0': round(tpr_0, 4),
        'tpr_group_1': round(tpr_1, 4)
    }

# PIMA fairness analysis (using Age as sensitive feature: <45 vs >=45)
pima_age_feature = (pima_X_test['Age'] >= 45).astype(int).values
pima_predictions = pima_model.predict(pima_X_test)
pima_fairness = calculate_fairness_metrics(pima_y_test, pima_predictions, pima_age_feature)

print("\nPIMA Dataset - Fairness Analysis (Sensitive: Age < 45 vs >= 45):")
print(f"  Demographic Parity Difference: {pima_fairness['demographic_parity_diff']}")
print(f"    Interpretation: Lower is better (< 0.1 is good)")
print(f"  Equalized Odds Difference: {pima_fairness['equalized_odds_diff']}")
print(f"    Interpretation: Lower indicates fair treatment across groups")
print(f"  Positive Prediction Rate (Age < 45): {pima_fairness['ppr_group_0']}")
print(f"  Positive Prediction Rate (Age >= 45): {pima_fairness['ppr_group_1']}")

# Heart Disease fairness analysis
heart_age_feature = (heart_X_test['age'] >= 50).astype(int).values
heart_predictions = heart_model.predict(heart_X_test)
heart_fairness = calculate_fairness_metrics(heart_y_test, heart_predictions, heart_age_feature)

print("\nHeart Disease Dataset - Fairness Analysis (Sensitive: Age < 50 vs >= 50):")
print(f"  Demographic Parity Difference: {heart_fairness['demographic_parity_diff']}")
print(f"  Equalized Odds Difference: {heart_fairness['equalized_odds_diff']}")
print(f"  Positive Prediction Rate (Age < 50): {heart_fairness['ppr_group_0']}")
print(f"  Positive Prediction Rate (Age >= 50): {heart_fairness['ppr_group_1']}")


ETHICAL ANALYSIS - FAIRNESS & BIAS DETECTION

PIMA Dataset - Fairness Analysis (Sensitive: Age < 45 vs >= 45):
  Demographic Parity Difference: 0.2239
    Interpretation: Lower is better (< 0.1 is good)
  Equalized Odds Difference: 0.5652
    Interpretation: Lower indicates fair treatment across groups
  Positive Prediction Rate (Age < 45): 0.2239
  Positive Prediction Rate (Age >= 45): 0

Heart Disease Dataset - Fairness Analysis (Sensitive: Age < 50 vs >= 50):
  Demographic Parity Difference: 0.4884
  Equalized Odds Difference: 0.9444
  Positive Prediction Rate (Age < 50): 0.4884
  Positive Prediction Rate (Age >= 50): 0


## 5. Privacy & Ethical Considerations

In [ ]:
print("\n" + "="*70)
print("PRIVACY & ETHICAL GUIDELINES")
print("="*70)

privacy_guidelines = """
\n1. DATA PRIVACY
   ✓ All personal health data must be encrypted (AES-256)
   ✓ HIPAA compliance required for US deployment
   ✓ Data retention: Delete after 6 months if not needed
   ✓ User consent: Explicit opt-in before data collection

2. MODEL TRANSPARENCY
   ✓ Users should understand how risk is calculated
   ✓ Confidence intervals on predictions provided
   ✓ Model limitations clearly communicated
   ✓ Regular model audits for performance drift

3. FAIRNESS & BIAS MITIGATION
   ✓ Monitor demographic parity regularly
   ✓ Retrain model quarterly with balanced data
   ✓ Investigate and mitigate identified biases
   ✓ Maintain separate fairness dashboards

4. ACCOUNTABILITY & LIABILITY
   ✓ AI agent serves as DECISION SUPPORT only
   ✓ Not a substitute for professional medical advice
   ✓ Always recommend consulting healthcare providers
   ✓ Clear liability disclaimers in user agreement

5. USER RIGHTS
   ✓ Right to access: Users can see their data
   ✓ Right to rectification: Correct inaccurate data
   ✓ Right to deletion: Request data removal (within legal limits)
   ✓ Right to explanation: Receive explainable AI outputs (LIME/SHAP)
"""

print(privacy_guidelines)

# Summary metrics
print("\n" + "="*70)
print("DEPLOYMENT READINESS CHECKLIST")
print("="*70)

checklist = {
    '✓ Data Preprocessing Pipeline': 'Complete - Notebook 1',
    '✓ Feature Engineering': 'Complete - Notebook 2',
    '✓ Supervised Model Training & Evaluation': 'Complete - Notebook 3',
    '✓ Explainability & Visualization': 'Complete - Notebook 4',
    '✓ AI Agent Implementation': 'Complete - Notebook 5',
    '✓ Fairness Analysis': 'Complete',
    '✓ Privacy Framework': 'Recommended',
    '✓ Model Monitoring': 'Recommended Pre-Deployment'
}

for item, status in checklist.items():
    print(f"{item}: {status}")


PRIVACY & ETHICAL GUIDELINES


1. DATA PRIVACY
   ✓ All personal health data must be encrypted (AES-256)
   ✓ HIPAA compliance required for US deployment
   ✓ Data retention: Delete after 6 months if not needed
   ✓ User consent: Explicit opt-in before data collection

2. MODEL TRANSPARENCY
   ✓ Users should understand how risk is calculated
   ✓ Confidence intervals on predictions provided
   ✓ Model limitations clearly communicated
   ✓ Regular model audits for performance drift

3. FAIRNESS & BIAS MITIGATION
   ✓ Monitor demographic parity regularly
   ✓ Retrain model quarterly with balanced data
   ✓ Investigate and mitigate identified biases
   ✓ Maintain separate fairness dashboards

4. ACCOUNTABILITY & LIABILITY
   ✓ AI agent serves as DECISION SUPPORT only
   ✓ Not a substitute for professional medical advice
   ✓ Always recommend consulting healthcare providers
   ✓ Clear liability disclaimers in user agreement

5. USER RIGHTS
   ✓ Right to access: Users can see their data
  